# LeetCode 2347 — Best Poker Hand bằng Python

Link đề: https://leetcode.com/problems/best-poker-hand/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Counting

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def bestHand(self, ranks: List[int], suits: List[str]) -> str:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: từ 5 lá bài (mỗi lá có **số (rank)** và **chất (suit)**), xác định **loại tay bài tốt nhất** theo thứ tự ưu tiên cố định.
2. Biết đếm tần suất bằng `collections.Counter`, bằng mảng đếm 14 ô, và bằng `set`; biết lấy số đếm lớn nhất bằng `max(counter.values())`.
3. Biết dùng `len(set(...))`, `all(...)`, `sorted` + `groupby`, và hiểu **thứ tự ưu tiên** khi nhiều điều kiện cùng đúng (kiểm tra từ mạnh tới yếu).
4. Viết 5 cách: `Counter`, mảng đếm 14 ô, `set` cho chất rồi đếm số, sắp xếp rồi nhìn đoạn liên tiếp, và bảng tra cứu theo số đếm lớn nhất.
5. Trace, test ngẫu nhiên đối chiếu, và tránh lỗi: dùng `len(set(ranks))` để suy ra Pair (sai với hai đôi), xếp hạng `Pair` trước `Three of a Kind`, quên rằng `Flush` có ưu tiên cao nhất, nhầm bốn lá cùng số.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `ranks` (số của lá bài) và mảng ký tự `suits` (chất của lá bài); lá thứ `i` có số `ranks[i]` và chất `suits[i]`. Có **đúng 5 lá bài**.

Các loại tay bài, **từ tốt nhất đến kém nhất**:

1. `"Flush"`: cả 5 lá **cùng chất**.
2. `"Three of a Kind"`: có **ít nhất 3 lá cùng số**.
3. `"Pair"`: có **2 lá cùng số** (và không có 3 lá cùng số).
4. `"High Card"`: 5 lá có số **đôi một khác nhau**.

Trả về chuỗi tên loại tay bài **tốt nhất** mà bạn có.

| Ví dụ | ranks | suits | Output | Giải thích |
|---|---|---|---|---|
| 1 | `[13,2,3,1,9]` | `["a","a","a","a","a"]` | `"Flush"` | cả 5 lá chất `a` |
| 2 | `[4,4,2,4,4]` | `["d","a","a","b","c"]` | `"Three of a Kind"` | có 4 lá số `4` (ít nhất 3 lá cùng số) |
| 3 | `[10,10,2,12,9]` | `["a","b","c","a","d"]` | `"Pair"` | có 2 lá số `10`, không đủ 3 |

Constraints: `ranks.length == suits.length == 5`; `1 <= ranks[i] <= 13`; `'a' <= suits[i] <= 'd'`; không có hai lá nào trùng cả số lẫn chất.

In [1]:
examples = [
    ([13, 2, 3, 1, 9], ["a", "a", "a", "a", "a"], "Flush"),
    ([4, 4, 2, 4, 4], ["d", "a", "a", "b", "c"], "Three of a Kind"),
    ([10, 10, 2, 12, 9], ["a", "b", "c", "a", "d"], "Pair"),
]
for ranks, suits, expected in examples:
    from collections import Counter
    print(f"ranks={ranks} suits={suits}: số chất khác nhau={len(set(suits))}, đếm số={dict(Counter(ranks))} -> mong đợi {expected!r}")

ranks=[13, 2, 3, 1, 9] suits=['a', 'a', 'a', 'a', 'a']: số chất khác nhau=1, đếm số={13: 1, 2: 1, 3: 1, 1: 1, 9: 1} -> mong đợi 'Flush'
ranks=[4, 4, 2, 4, 4] suits=['d', 'a', 'a', 'b', 'c']: số chất khác nhau=4, đếm số={4: 4, 2: 1} -> mong đợi 'Three of a Kind'
ranks=[10, 10, 2, 12, 9] suits=['a', 'b', 'c', 'a', 'd']: số chất khác nhau=4, đếm số={10: 2, 2: 1, 12: 1, 9: 1} -> mong đợi 'Pair'


## 3. Kiến thức nền

### 3.1. `set` — tập hợp các phần tử khác nhau

`set(iterable)` giữ mỗi giá trị **một lần**. `len(set(suits))` là số chất **khác nhau**. Nếu `len(set(suits)) == 1` thì mọi lá cùng chất.

In [2]:
suits = ["a", "a", "a", "a", "a"]
print(set(suits), len(set(suits)))
print(len(set(["a", "b", "a", "c", "a"])))

{'a'} 1
3


### 3.2. `collections.Counter` — đếm tần suất

`Counter(iterable)` trả từ điển `{giá trị: số lần}`. `.values()` cho các số lần; `max(cnt.values())` là số lần xuất hiện **nhiều nhất** của một giá trị — đúng thứ ta cần để biết có `Pair` hay `Three of a Kind`.

In [3]:
from collections import Counter

cnt = Counter([10, 10, 2, 12, 9])
print(cnt)
print(list(cnt.values()), max(cnt.values()))
print(Counter([4, 4, 2, 4, 4]).most_common(1))

Counter({10: 2, 2: 1, 12: 1, 9: 1})
[2, 1, 1, 1] 2
[(4, 4)]


### 3.3. Mảng đếm 14 ô

Số lá nằm trong `1..13`. Mảng `cnt = [0] * 14` (bỏ ô 0) với `cnt[r] += 1` đếm không cần băm, chỉ số bằng chính số của lá bài.

In [4]:
ranks = [10, 10, 2, 12, 9]
cnt = [0] * 14
for r in ranks:
    cnt[r] += 1
print(cnt)
print(max(cnt))

[0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 2, 0, 1, 0]
2


### 3.4. `all(...)` và so sánh dây chuyền

`all(s == suits[0] for s in suits)` đúng khi mọi chất bằng chất đầu. Tương đương `len(set(suits)) == 1`. Cũng có thể viết `suits[0] == suits[1] == ... == suits[4]`.

In [5]:
suits = ["a", "a", "b", "a", "a"]
print(all(s == suits[0] for s in suits))
print(len(set(suits)) == 1)
print(suits[0] == suits[1] == suits[2] == suits[3] == suits[4])

False
False
False


### 3.5. Thứ tự ưu tiên của các điều kiện

Nhiều điều kiện có thể cùng đúng. Ví dụ `[4,4,4,4,2]` có 4 lá cùng số, thì **cũng** thỏa "có 2 lá cùng số" (`Pair`) và "có ít nhất 3 lá cùng số". Quy tắc: kiểm tra **từ loại mạnh tới loại yếu** và trả về ngay khi đúng.

In [6]:
def classify(max_count, is_flush):
    if is_flush:
        return "Flush"
    if max_count >= 3:
        return "Three of a Kind"
    if max_count == 2:
        return "Pair"
    return "High Card"

for mc, fl in ((1, False), (2, False), (3, False), (4, False), (2, True)):
    print(f"max_count={mc}, flush={fl} -> {classify(mc, fl)}")

max_count=1, flush=False -> High Card
max_count=2, flush=False -> Pair
max_count=3, flush=False -> Three of a Kind
max_count=4, flush=False -> Three of a Kind
max_count=2, flush=True -> Flush


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Chỉ cần **hai con số** từ 5 lá bài:

- `flush`: tất cả chất giống nhau hay không (đếm số chất khác nhau).
- `mx`: **số lá cùng số nhiều nhất** (số lần xuất hiện lớn nhất trong `ranks`).

Sau đó đọc bảng theo thứ tự ưu tiên:

| Điều kiện | Kết quả |
|---|---|
| `flush` đúng | `"Flush"` |
| `mx >= 3` | `"Three of a Kind"` |
| `mx == 2` | `"Pair"` |
| còn lại (`mx == 1`) | `"High Card"` |

Tính tay `[4,4,2,4,4]`, chất `d,a,a,b,c`: chất khác nhau nhiều -> không Flush; số `4` xuất hiện 4 lần -> `mx = 4 >= 3` -> `"Three of a Kind"`.

Lưu ý: `mx` cần **cao nhất** không phải số loại số khác nhau; hai đôi `[3,3,5,5,9]` có 3 số khác nhau nhưng `mx = 2` (vẫn là `Pair`).

In [7]:
from collections import Counter

for ranks, suits in (([4, 4, 2, 4, 4], ["d", "a", "a", "b", "c"]), ([3, 3, 5, 5, 9], ["a", "b", "c", "d", "a"])):
    flush = len(set(suits)) == 1
    mx = max(Counter(ranks).values())
    print(ranks, "flush =", flush, "| max count =", mx, "| số giá trị khác nhau =", len(set(ranks)))

[4, 4, 2, 4, 4] flush = False | max count = 4 | số giá trị khác nhau = 2
[3, 3, 5, 5, 9] flush = False | max count = 2 | số giá trị khác nhau = 3


## 5. Cách 1 — Brute Force / baseline: `Counter` và kiểm tra theo thứ tự

**Ý tưởng:** nếu mọi chất bằng nhau -> `Flush`. Ngược lại đếm số với `Counter`, lấy `max` của các số đếm, rồi so với `3`, `2`.

**Độ phức tạp:** `O(5)` = `O(n)` với `n = 5` lá. Với đề cố định 5 lá thì mọi cách đều hằng số; **nút thắt** duy nhất là độ rõ ràng và tránh sai điều kiện.

In [8]:
from typing import List
from collections import Counter

class SolutionCounter:
    def bestHand(self, ranks: List[int], suits: List[str]) -> str:
        if len(set(suits)) == 1:
            return "Flush"
        mx = max(Counter(ranks).values())
        if mx >= 3:
            return "Three of a Kind"
        if mx == 2:
            return "Pair"
        return "High Card"

for r, s in (([13, 2, 3, 1, 9], list("aaaaa")), ([4, 4, 2, 4, 4], list("daabc")), ([10, 10, 2, 12, 9], list("abcad"))):
    print(r, s, "->", SolutionCounter().bestHand(r, s))

[13, 2, 3, 1, 9] ['a', 'a', 'a', 'a', 'a'] -> Flush
[4, 4, 2, 4, 4] ['d', 'a', 'a', 'b', 'c'] -> Three of a Kind
[10, 10, 2, 12, 9] ['a', 'b', 'c', 'a', 'd'] -> Pair


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** loại tay bài chỉ phụ thuộc `flush` (có/không) và `mx` (số lần xuất hiện lớn nhất của một số). Không cần biết số nào, chất nào, và không cần so sánh từng cặp lá.

**Invariant:** sau khi đếm, `mx = max_r count(r)` là số lá cùng số nhiều nhất; `mx >= 3` nghĩa là tồn tại số xuất hiện ít nhất 3 lần.

**Quy tắc:** ưu tiên `Flush` > `mx >= 3` > `mx == 2` > còn lại. Vì chỉ có 5 lá nên `mx` chỉ nhận `1..5`; `mx = 3, 4, 5` đều là "Three of a Kind" (4 hoặc 5 lá cùng số cũng được).

Lưu ý **không** dùng `len(set(ranks))` để suy ra loại: `len == 4` suy ra Pair đúng, nhưng `len == 3` có thể là Three of a Kind (`3,3,3,1,2`) **hoặc** hai đôi (`3,3,5,5,9`, vẫn là `Pair`).

## 7. Các cách tiếp theo

### Cách 2 — Mảng đếm 14 ô

`cnt[r]` đếm số lá có số `r`. `max(cnt)` là `mx`. Không cần `Counter`.

In [9]:
from typing import List

class SolutionArray:
    def bestHand(self, ranks: List[int], suits: List[str]) -> str:
        if all(s == suits[0] for s in suits):
            return "Flush"
        cnt = [0] * 14
        for r in ranks:
            cnt[r] += 1
        mx = max(cnt)
        if mx >= 3:
            return "Three of a Kind"
        if mx == 2:
            return "Pair"
        return "High Card"

for r, s in (([13, 2, 3, 1, 9], list("aaaaa")), ([4, 4, 2, 4, 4], list("daabc")), ([1, 2, 3, 4, 5], list("abcad"))):
    print(r, s, "->", SolutionArray().bestHand(r, s))

[13, 2, 3, 1, 9] ['a', 'a', 'a', 'a', 'a'] -> Flush
[4, 4, 2, 4, 4] ['d', 'a', 'a', 'b', 'c'] -> Three of a Kind
[1, 2, 3, 4, 5] ['a', 'b', 'c', 'a', 'd'] -> High Card


**Độ phức tạp Cách 2:** Time `O(n)`, Space `O(1)` (mảng 14 ô).

### Cách 3 — Dùng `count` trên list, thử từng giá trị

Với mỗi lá, `ranks.count(r)` cho số lá cùng số. Lấy `max(ranks.count(r) for r in ranks)`. Mỗi `count` quét 5 phần tử: `O(5 * 5)` — vẫn hằng số nhưng "bậc hai" theo `n`.

In [10]:
from typing import List

class SolutionCount:
    def bestHand(self, ranks: List[int], suits: List[str]) -> str:
        if suits.count(suits[0]) == len(suits):
            return "Flush"
        mx = max(ranks.count(r) for r in ranks)
        return "Three of a Kind" if mx >= 3 else "Pair" if mx == 2 else "High Card"

for r, s in (([13, 2, 3, 1, 9], list("aaaaa")), ([4, 4, 2, 4, 4], list("daabc")), ([10, 10, 2, 12, 9], list("abcad"))):
    print(r, s, "->", SolutionCount().bestHand(r, s))

[13, 2, 3, 1, 9] ['a', 'a', 'a', 'a', 'a'] -> Flush
[4, 4, 2, 4, 4] ['d', 'a', 'a', 'b', 'c'] -> Three of a Kind
[10, 10, 2, 12, 9] ['a', 'b', 'c', 'a', 'd'] -> Pair


**Độ phức tạp Cách 3:** Time `O(n^2)` (với `n = 5`, hằng số), Space `O(1)`. Biểu thức điều kiện lồng `a if x else b if y else c` gọn nhưng khó đọc; dùng cẩn thận.

### Cách 4 — Sắp xếp rồi tìm đoạn liên tiếp dài nhất

Sắp xếp `ranks` rồi đếm độ dài **đoạn liên tiếp bằng nhau** dài nhất (`itertools.groupby`). Độ dài đó chính là `mx`.

In [11]:
from typing import List
from itertools import groupby

class SolutionSortRuns:
    def bestHand(self, ranks: List[int], suits: List[str]) -> str:
        if len(set(suits)) == 1:
            return "Flush"
        mx = max(len(list(g)) for _, g in groupby(sorted(ranks)))
        if mx >= 3:
            return "Three of a Kind"
        return "Pair" if mx == 2 else "High Card"

for r, s in (([13, 2, 3, 1, 9], list("aaaaa")), ([4, 4, 2, 4, 4], list("daabc")), ([3, 3, 5, 5, 9], list("abcda"))):
    print(r, s, "->", SolutionSortRuns().bestHand(r, s))

[13, 2, 3, 1, 9] ['a', 'a', 'a', 'a', 'a'] -> Flush
[4, 4, 2, 4, 4] ['d', 'a', 'a', 'b', 'c'] -> Three of a Kind
[3, 3, 5, 5, 9] ['a', 'b', 'c', 'd', 'a'] -> Pair


**Độ phức tạp Cách 4:** Time `O(n log n)` (sắp xếp), Space `O(n)`. Với `n = 5` là hằng số. Cách này hữu ích khi bài tương tự mở rộng sang **sảnh (straight)**: sau khi sắp xếp ta kiểm tra các số liên tiếp.

### Cách 5 — Bảng tra cứu theo `mx`

Thay chuỗi `if` bằng danh sách `NAMES = [None, "High Card", "Pair", "Three of a Kind", "Three of a Kind", "Three of a Kind"]` rồi `NAMES[mx]`. Gọn và hết lo thứ tự `if`.

In [12]:
from typing import List
from collections import Counter

class SolutionTable:
    NAMES = [None, "High Card", "Pair", "Three of a Kind", "Three of a Kind", "Three of a Kind"]

    def bestHand(self, ranks: List[int], suits: List[str]) -> str:
        if len(set(suits)) == 1:
            return "Flush"
        return self.NAMES[max(Counter(ranks).values())]

for r, s in (([13, 2, 3, 1, 9], list("aaaaa")), ([4, 4, 2, 4, 4], list("daabc")), ([10, 10, 2, 12, 9], list("abcad"))):
    print(r, s, "->", SolutionTable().bestHand(r, s))

[13, 2, 3, 1, 9] ['a', 'a', 'a', 'a', 'a'] -> Flush
[4, 4, 2, 4, 4] ['d', 'a', 'a', 'b', 'c'] -> Three of a Kind
[10, 10, 2, 12, 9] ['a', 'b', 'c', 'a', 'd'] -> Pair


**Độ phức tạp Cách 5:** Time `O(n)`, Space `O(1)`. `mx` chỉ là `1..5` nên `NAMES` có 6 ô là đủ.

## 8. TRACE từng bước

Trace Cách 1 với `ranks = [10,10,2,12,9]`, `suits = ["a","b","c","a","d"]`:

| Bước | Giá trị | Kết luận |
|---|---|---|
| 1. Chất | `set(suits) = {a,b,c,d}`, `len = 4` | không Flush |
| 2. Đếm số | `Counter = {10:2, 2:1, 12:1, 9:1}` | |
| 3. `mx` | `max(values) = 2` | |
| 4. `mx >= 3`? | không | |
| 5. `mx == 2`? | có | **Pair** |

In [13]:
from collections import Counter

def trace_counter(ranks, suits):
    print(f"ranks={ranks}, suits={suits}")
    kinds = set(suits)
    print(f"  chất khác nhau: {sorted(kinds)} -> {len(kinds)}")
    if len(kinds) == 1:
        print("  -> Flush")
        return "Flush"
    cnt = Counter(ranks)
    mx = max(cnt.values())
    print(f"  Counter={dict(cnt)}, mx={mx}")
    for name, ok in (("Three of a Kind", mx >= 3), ("Pair", mx == 2)):
        print(f"  kiểm tra {name}: {ok}")
        if ok:
            return name
    print("  -> High Card")
    return "High Card"

print("kết quả:", trace_counter([10, 10, 2, 12, 9], ["a", "b", "c", "a", "d"]))
print("kết quả:", trace_counter([4, 4, 2, 4, 4], ["d", "a", "a", "b", "c"]))

ranks=[10, 10, 2, 12, 9], suits=['a', 'b', 'c', 'a', 'd']
  chất khác nhau: ['a', 'b', 'c', 'd'] -> 4
  Counter={10: 2, 2: 1, 12: 1, 9: 1}, mx=2
  kiểm tra Three of a Kind: False
  kiểm tra Pair: True
kết quả: Pair
ranks=[4, 4, 2, 4, 4], suits=['d', 'a', 'a', 'b', 'c']
  chất khác nhau: ['a', 'b', 'c', 'd'] -> 4
  Counter={4: 4, 2: 1}, mx=4
  kiểm tra Three of a Kind: True
kết quả: Three of a Kind


In [14]:
def trace_array(ranks):
    cnt = [0] * 14
    for r in ranks:
        cnt[r] += 1
        nz = {i: c for i, c in enumerate(cnt) if c}
        print(f"  thêm {r}: các ô khác 0 = {nz}")
    return max(cnt)

print("mx =", trace_array([3, 3, 5, 5, 9]))

  thêm 3: các ô khác 0 = {3: 1}
  thêm 3: các ô khác 0 = {3: 2}
  thêm 5: các ô khác 0 = {3: 2, 5: 1}
  thêm 5: các ô khác 0 = {3: 2, 5: 2}
  thêm 9: các ô khác 0 = {3: 2, 5: 2, 9: 1}
mx = 2


## 9. Code nộp LeetCode

In [15]:
from typing import List
from collections import Counter

class Solution:
    def bestHand(self, ranks: List[int], suits: List[str]) -> str:
        if len(set(suits)) == 1:
            return "Flush"
        mx = max(Counter(ranks).values())
        if mx >= 3:
            return "Three of a Kind"
        if mx == 2:
            return "Pair"
        return "High Card"

print(Solution().bestHand([13, 2, 3, 1, 9], ["a", "a", "a", "a", "a"]))
print(Solution().bestHand([4, 4, 2, 4, 4], ["d", "a", "a", "b", "c"]))
print(Solution().bestHand([10, 10, 2, 12, 9], ["a", "b", "c", "a", "d"]))

Flush
Three of a Kind
Pair


## 10. Vì sao thuật toán đúng

- **Đủ thông tin.** Điều kiện của bốn loại chỉ dùng: "cả 5 chất bằng nhau" và "có ít nhất `k` lá cùng số" (`k = 3`, `2`). Hai đại lượng `flush` và `mx` quyết định mọi điều kiện, vì "tồn tại số xuất hiện ít nhất `k` lần" tương đương `mx >= k`.
- **Thứ tự ưu tiên.** Ta kiểm tra từ tốt nhất tới kém nhất và trả về ngay; nên khi nhiều điều kiện cùng đúng, kết quả là điều kiện **mạnh nhất** — đúng đề ("tốt nhất").
- **`mx >= 3` thay vì `== 3`.** Đề định nghĩa Three of a Kind là "**ít nhất** 3 lá cùng số", nên 4 hoặc 5 lá cùng số vẫn là Three of a Kind. (Đề đảm bảo không có hai lá trùng cả số lẫn chất, vì chỉ có 4 chất nên tối đa 4 lá cùng số; `5` không xảy ra trong dữ liệu hợp lệ.)
- **High Card.** Không Flush và `mx == 1` nghĩa là 5 số đôi một khác nhau — đúng định nghĩa.

## 11. Phân tích Time / Space Complexity

Gọi `n = 5` (số lá bài, cố định).

| Cách | Time | Space |
|---|---|---|
| 1. `Counter` | `O(n)` | `O(1)` |
| 2. Mảng 14 ô | `O(n)` | `O(1)` |
| 3. `list.count` | `O(n^2)` | `O(1)` |
| 4. Sắp xếp + `groupby` | `O(n log n)` | `O(n)` |
| 5. Bảng tra cứu | `O(n)` | `O(1)` |

Vì `n = 5` cố định nên mọi cách thực chất là `O(1)`. Chọn cách rõ nhất; khi mở rộng số lá, cách 1/2/5 giữ `O(n)`, cách 3 trở thành `O(n^2)`.

## 12. Bộ test

Đối chiếu tất cả các cách trên ví dụ đề, edge case và test ngẫu nhiên có `seed`.

In [16]:
import random, itertools

solutions = [SolutionCounter(), SolutionArray(), SolutionCount(), SolutionSortRuns(), SolutionTable(), Solution()]

fixed = [
    ([13, 2, 3, 1, 9], list("aaaaa"), "Flush"),
    ([4, 4, 2, 4, 4], list("daabc"), "Three of a Kind"),
    ([10, 10, 2, 12, 9], list("abcad"), "Pair"),
    ([1, 2, 3, 4, 5], list("abcdd"), "High Card"),
    ([1, 1, 1, 1, 1], list("aaaaa"), "Flush"),            # Flush ưu tiên hơn cả bốn lá cùng số
    ([7, 7, 7, 2, 9], list("abcda"), "Three of a Kind"),  # đúng ba lá
    ([3, 3, 5, 5, 9], list("abcda"), "Pair"),              # hai đôi vẫn là Pair
    ([3, 3, 3, 5, 5], list("abcda"), "Three of a Kind"),   # full house trong poker thật, ở đây là Three
    ([2, 2, 2, 2, 9], list("abcdd"), "Three of a Kind"),   # bốn lá cùng số
    ([13, 12, 11, 10, 9], list("aabbb"), "High Card"),
]
for ranks, suits, expected in fixed:
    for s in solutions:
        assert s.bestHand(list(ranks), list(suits)) == expected, (type(s).__name__, ranks, suits, expected)
print("đã qua", len(fixed), "test cố định cho", len(solutions), "cách")

đã qua 10 test cố định cho 6 cách


In [17]:
random.seed(2347)
deck = [(r, s) for r in range(1, 14) for s in "abcd"]

def truth(ranks, suits):
    # định nghĩa trực tiếp: đếm các cặp lá cùng số
    if len(set(suits)) == 1:
        return "Flush"
    same = max(sum(1 for r2 in ranks if r2 == r) for r in ranks)
    return "Three of a Kind" if same >= 3 else "Pair" if same == 2 else "High Card"

seen = set()
for _ in range(3000):
    hand = random.sample(deck, 5)
    ranks = [r for r, _ in hand]
    suits = [s for _, s in hand]
    ref = truth(ranks, suits)
    seen.add(ref)
    for s in solutions:
        assert s.bestHand(list(ranks), list(suits)) == ref, (type(s).__name__, hand)
print("3000 tay bài ngẫu nhiên đối chiếu: OK; các loại đã gặp:", sorted(seen))

3000 tay bài ngẫu nhiên đối chiếu: OK; các loại đã gặp: ['Flush', 'High Card', 'Pair', 'Three of a Kind']


In [18]:
# Thống kê: tần suất từng loại trong 20000 tay bài ngẫu nhiên (để thấy High Card/Pair phổ biến nhất)
from collections import Counter
random.seed(5)
stat = Counter()
for _ in range(20000):
    hand = random.sample(deck, 5)
    stat[Solution().bestHand([r for r, _ in hand], [s for _, s in hand])] += 1
print(dict(stat))
assert stat["High Card"] > stat["Pair"] > stat["Three of a Kind"] or stat["Pair"] > stat["High Card"] > stat["Three of a Kind"]
assert set(stat) == {"Flush", "Three of a Kind", "Pair", "High Card"}

{'High Card': 10070, 'Pair': 9374, 'Three of a Kind': 515, 'Flush': 41}


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Suy ra loại bài từ `len(set(ranks))`

`len(set(ranks)) == 4` thì chắc chắn có một cặp. Nhưng `len == 3` mơ hồ: có thể `Three of a Kind` (`3,3,3,1,2`) hoặc **hai đôi** (`3,3,5,5,9` là `Pair`).

In [19]:
def sai_set(ranks, suits):
    if len(set(suits)) == 1:
        return "Flush"
    k = len(set(ranks))
    return {5: "High Card", 4: "Pair", 3: "Three of a Kind"}.get(k, "Three of a Kind")   # SAI

for r in ([3, 3, 3, 1, 2], [3, 3, 5, 5, 9]):
    print(r, "sai:", sai_set(r, list("abcda")), "| đúng:", Solution().bestHand(r, list("abcda")))

[3, 3, 3, 1, 2] sai: Three of a Kind | đúng: Three of a Kind
[3, 3, 5, 5, 9] sai: Three of a Kind | đúng: Pair


### Lỗi 2: Kiểm tra `Pair` trước `Three of a Kind` (đảo thứ tự ưu tiên)

Nếu `mx >= 2` trả `Pair` trước thì `[4,4,4,2,9]` cho `Pair` thay vì `Three of a Kind`.

In [20]:
from collections import Counter

def sai_thu_tu(ranks, suits):
    if len(set(suits)) == 1:
        return "Flush"
    mx = max(Counter(ranks).values())
    if mx >= 2:                       # SAI: Pair đứng trước Three
        return "Pair"
    if mx >= 3:
        return "Three of a Kind"
    return "High Card"

print("sai :", sai_thu_tu([4, 4, 4, 2, 9], list("abcda")), "| đúng:", Solution().bestHand([4, 4, 4, 2, 9], list("abcda")))

sai : Pair | đúng: Three of a Kind


### Lỗi 3: Dùng `mx == 3` thay vì `mx >= 3`

Với bốn lá cùng số (`mx == 4`) đề vẫn yêu cầu `Three of a Kind`. Dùng `== 3` làm rớt xuống nhánh `Pair`/`High Card`.

In [21]:
from collections import Counter

def sai_bang_ba(ranks, suits):
    if len(set(suits)) == 1:
        return "Flush"
    mx = max(Counter(ranks).values())
    if mx == 3:                       # SAI: bỏ sót 4 lá cùng số
        return "Three of a Kind"
    if mx == 2:
        return "Pair"
    return "High Card"

print("sai :", sai_bang_ba([4, 4, 2, 4, 4], list("dabbc")), "| đúng:", Solution().bestHand([4, 4, 2, 4, 4], list("dabbc")))

sai : High Card | đúng: Three of a Kind


### Lỗi 4: Kiểm tra `Flush` sau cùng (hoặc quên)

`Flush` có ưu tiên cao nhất. Nếu kiểm tra Flush sau `Three of a Kind` thì `[4,4,4,2,9]` cùng chất lại ra `Three` thay vì `Flush`.

In [22]:
from collections import Counter

def sai_flush_cuoi(ranks, suits):
    mx = max(Counter(ranks).values())
    if mx >= 3:
        return "Three of a Kind"
    if mx == 2:
        return "Pair"
    if len(set(suits)) == 1:           # SAI: Flush phải kiểm tra đầu tiên
        return "Flush"
    return "High Card"

print("sai :", sai_flush_cuoi([4, 4, 4, 2, 9], list("aaaaa")), "| đúng:", Solution().bestHand([4, 4, 4, 2, 9], list("aaaaa")))

sai : Three of a Kind | đúng: Flush


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề xếp hạng nhiều loại theo ưu tiên -> kiểm tra **từ mạnh tới yếu**, `return` ngay khi đúng.
- "Có ít nhất `k` phần tử bằng nhau" -> `max(Counter(x).values()) >= k`.
- "Tất cả bằng nhau" -> `len(set(x)) == 1`.
- Số trong khoảng nhỏ (`1..13`) -> mảng đếm thay `Counter`.
- Cẩn thận khi suy loại từ số phần tử **khác nhau**: dễ nhầm hai đôi với bộ ba.
- Cùng pattern: Majority Element, Contains Duplicate, Valid Anagram (đếm tần suất).

## 15. Biến thể của bài

- **Thêm loại tay bài**: Two Pair, Straight (sảnh), Full House, Four of a Kind — cần xét **dạng của các số đếm** (ví dụ sắp xếp các số đếm giảm dần: `[2,2,1]` là Two Pair, `[3,2]` là Full House).
- **Chất có nhiều hơn 4 loại hoặc số lá khác 5.**
- **So sánh hai tay bài** để tìm người thắng (cần xếp hạng chi tiết hơn).
- **Tay bài tốt nhất từ 7 lá** (Texas Hold'em): thử mọi tổ hợp 5 trong 7 (`itertools.combinations`).
- **Đếm số tay bài mỗi loại** trên cả bộ bài 52 lá (như ô thống kê ở Mục 12).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `Counter` | `max` số đếm, `if` theo ưu tiên | `O(n)` | `O(1)` | cách chuẩn, rõ ràng |
| 2. Mảng 14 ô | đếm theo chỉ số số | `O(n)` | `O(1)` | không cần thư viện |
| 3. `list.count` | đếm từng lá | `O(n^2)` | `O(1)` | `n` rất nhỏ, code ngắn |
| 4. Sắp xếp + `groupby` | đoạn bằng nhau dài nhất | `O(n log n)` | `O(n)` | mở rộng sang sảnh |
| 5. Bảng tra cứu | `NAMES[mx]` | `O(n)` | `O(1)` | tránh chuỗi `if` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, dự đoán `Solution().bestHand([5, 5, 5, 5, 9], ["a", "b", "c", "d", "a"])` và `Solution().bestHand([1, 2, 3, 4, 5], ["a","a","a","a","a"])`.

**Bài 2 (tính tay).** Với `ranks = [8, 8, 3, 3, 3]`, ghi `Counter` và `mx`; loại tay bài theo đề là gì?

**Bài 3 (điền TODO).** Hoàn thành `max_same(ranks)` trả về số lá cùng số nhiều nhất bằng mảng đếm 14 ô (cell dưới chưa được gọi).

**Bài 4 (tự viết lại).** Viết `is_straight(ranks)` kiểm tra 5 số có tạo thành **sảnh** (5 số liên tiếp, bỏ qua trường hợp A-cao).

**Bài 5 (mini challenge).** Viết `hand_name(ranks, suits)` mở rộng thêm `"Two Pair"` và `"Full House"` (dùng danh sách số đếm đã sắp giảm dần).

In [23]:
# TODO Bài 3: điền vào chỗ trống, KHÔNG gọi hàm trong cell này
def max_same_todo(ranks):
    cnt = [0] * 14
    # for r in ranks: cnt[r] += 1
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: bốn lá số `5` -> `mx = 4 >= 3` nên `Three of a Kind`; tay thứ hai cùng chất nên `Flush`. Bài 2: `Counter = {8:2, 3:3}`, `mx = 3`.
- **Mức 2 (vừa):** Bài 3: `max(cnt)`. Bài 4: sắp xếp, 5 số khác nhau và `max - min == 4`.
- **Mức 3 (sâu):** Bài 5: `counts = sorted(Counter(ranks).values(), reverse=True)`; `[3,2]` -> Full House, `[2,2,1]` -> Two Pair; nhớ vẫn kiểm tra Flush trước và thêm các loại còn lại theo bài gốc.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
from collections import Counter

# Bài 1 và 2
assert Solution().bestHand([5, 5, 5, 5, 9], list("abcda")) == "Three of a Kind"
assert Solution().bestHand([1, 2, 3, 4, 5], list("aaaaa")) == "Flush"
c = Counter([8, 8, 3, 3, 3])
assert dict(c) == {8: 2, 3: 3} and max(c.values()) == 3
assert Solution().bestHand([8, 8, 3, 3, 3], list("abcda")) == "Three of a Kind"

# Bài 3
def max_same(ranks):
    cnt = [0] * 14
    for r in ranks:
        cnt[r] += 1
    return max(cnt)

# Bài 4
def is_straight(ranks):
    s = sorted(ranks)
    return len(set(s)) == 5 and s[-1] - s[0] == 4

# Bài 5
def hand_name(ranks, suits):
    if len(set(suits)) == 1:
        return "Flush"
    counts = sorted(Counter(ranks).values(), reverse=True)
    if counts[0] >= 4:
        return "Four of a Kind"
    if counts == [3, 2]:
        return "Full House"
    if counts[0] == 3:
        return "Three of a Kind"
    if counts == [2, 2, 1]:
        return "Two Pair"
    if counts[0] == 2:
        return "Pair"
    return "Straight" if is_straight(ranks) else "High Card"

assert max_same([4, 4, 2, 4, 4]) == 4 and max_same([1, 2, 3, 4, 5]) == 1
assert is_straight([3, 4, 5, 6, 7]) and not is_straight([3, 4, 5, 6, 8]) and not is_straight([2, 2, 3, 4, 5])
assert hand_name([3, 3, 5, 5, 9], list("abcda")) == "Two Pair"
assert hand_name([3, 3, 3, 5, 5], list("abcda")) == "Full House"
assert hand_name([2, 2, 2, 2, 9], list("abcdd")) == "Four of a Kind"
assert hand_name([3, 4, 5, 6, 7], list("abcda")) == "Straight"
assert hand_name([10, 10, 2, 12, 9], list("abcad")) == "Pair"
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (Flush trước, rồi `mx` theo ưu tiên)

```python
class Solution:
    def bestHand(self, ranks: List[int], suits: List[str]) -> str:
        if len(set(suits)) == 1:
            return "Flush"
        mx = max(Counter(ranks).values())
        if mx >= 3:
            return "Three of a Kind"
        if mx == 2:
            return "Pair"
        return "High Card"
```

**Checklist**

- [ ] Kiểm tra `Flush` **đầu tiên** (ưu tiên cao nhất)?
- [ ] Dùng `mx >= 3` (cả 4 lá cùng số cũng là Three of a Kind)?
- [ ] `Three of a Kind` được kiểm tra **trước** `Pair`?
- [ ] Không suy loại bài từ `len(set(ranks))` (hai đôi bị nhầm)?
- [ ] Đã thử hai đôi `[3,3,5,5,9]`, bốn lá `[4,4,2,4,4]`, và cả năm cùng chất?
- [ ] Trả đúng chuỗi (viết hoa/khoảng trắng): `"Three of a Kind"`, `"High Card"`?
- [ ] Signature đúng `bestHand(self, ranks: List[int], suits: List[str]) -> str`?